# RAG - v5 Build

**All features from v4 Build:**
- Refactored code structure
- Conversation Memory
- Parent Document Retriever

**NEW IN v5 Build:**
- **Multi-Query Retrieval**: Generate query variations for better coverage
- **FlashRank Reranking**: Cross-encoder reranking for better relevance

## Install Dependencies

In [ ]:
%pip install "langchain==0.3.27" -qqq
%pip install "langchain-community==0.3.31" -qqq
%pip install "langchain-openai==0.3.35" -qqq
%pip install "langchain-chroma==0.2.6" -qqq
%pip install pypdf -qqq
%pip install gradio -qqq
#--------------------------------------------------------------------------------
# NEW IN v5 Build: Install FlashRank for reranking
#--------------------------------------------------------------------------------
%pip install flashrank -qqq

## Configuration

## Load Secrets Function


In [ ]:
def load_secrets(keys):
    import os

    try:
        from google.colab import userdata  # type: ignore
        values = {key: userdata.get(key) for key in keys}
        in_colab = True
    except Exception:
        # Load from .env file for local development
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            raise ImportError(
                "python-dotenv is required for local development. "
                "Install it with: pip install python-dotenv"
            )
        
        # Get values from environment (loaded from .env file)
        values = {key: os.getenv(key) for key in keys}
        in_colab = False

    missing = [key for key, value in values.items() if not value]
    if missing:
        if in_colab:
            raise ValueError(f"Missing keys: {', '.join(missing)}. Set them in Colab Secrets.")
        else:
            env_file_help = (
                f"Missing keys: {', '.join(missing)}.\n\n"
                "Create a .env file in the project root with:\n"
                + "\n".join([f"{key}=YOUR_VALUE_HERE" for key in missing])
            )
            raise ValueError(env_file_help)

    for key, value in values.items():
        os.environ[key] = value

    return values

In [ ]:
try:
    secrets = load_secrets(['CHROMA_API_KEY', 'CHROMA_TENANT', 'CHROMA_DATABASE', 'OPENAI_API_KEY'])
except ValueError as e:
    print(e)


## Global Variables

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Version Management
VERSION = "v5"

# Vector Database collection name
COLLECTION_NAME = f"bitcoin_docs_{VERSION}"

# Embeddings Model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

# Classification LLM
classification_llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

# Vector Database
vectorstore = Chroma(
  embedding_function=embeddings,
  collection_name=COLLECTION_NAME,
  chroma_cloud_api_key=secrets["CHROMA_API_KEY"],
  tenant=secrets["CHROMA_TENANT"],
  database=secrets["CHROMA_DATABASE"]
)

# Base retriever from vectorstore
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

## Ingestion

In [ ]:
#@title Clean Text Function

import re

def clean_text(text: str) -> str:
    """
    TEXT PREPROCESSING - Clean PDF

    Args:
        text (str): Raw PDF text

    Returns:
        str: Cleaned text
    """
    # Remove multiple whitespaces
    text = re.sub(r'\s+', ' ', text)

    # Remove standalone page numbers
    text = re.sub(r'(?<=\.)\s*\d+\s*$', '', text)

    # Remove whitespace from beginning and end
    text = text.strip()

    return text

In [ ]:
#@title Topic Detection Function

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def detect_document_topic(documents: list) -> str:
    """
    DETECT DOCUMENT TOPIC

    Args:
        documents (list): List of Document objects from PyPDFLoader

    Returns:
        str: Detected topic (e.g., 'bitcoin', 'ethereum')
    """

    topic_detection_template = ChatPromptTemplate.from_template(
      """
      Analyze the following document content and determine its primary topic.

      Document content:
      {content}

      Based on this content, what is the primary topic? Answer with a single word or short phrase (e.g., 'bitcoin', 'ethereum').

      Examples:
      If the document is about Bitcoin, answer: bitcoin
      If the document is about Ethereum, answer: ethereum
      If the document is about general crypto technology, answer: crypto

      Primary topic:
      """
    )

    topic_detection_chain = topic_detection_template | classification_llm | StrOutputParser()

    # Extract sample content from first few pages
    sample_content = ""
    for doc in documents[:3]:
        sample_content += doc.page_content + "\n\n"

    # Limit to 4000 characters to save costs
    sample_content = sample_content[:4000]

    detected_topic = topic_detection_chain.invoke({
        "content": sample_content
    }).strip().lower()

    return detected_topic

In [ ]:
#@title Ingest Function

from langchain.document_loaders import PyPDFLoader

def ingest_documents(document_path: str):
    """
    INGESTION PIPELINE - VERSION v5 Build

    NEW IN v5 Build:
    - Uses vector store as base retriever

    Args:
        document_path (str): URL or local path to PDF

    Returns:
        tuple: (num_pages, detected_topic)
    """

    print("-" * 80)
    print(f"STARTING INGESTION PIPELINE - VERSION {VERSION} Build")
    print("-" * 80)

    #--------------------------------------------------------------------------------
    # STEP 1: LOAD DOCUMENTS
    #--------------------------------------------------------------------------------
    print("\n[1/6] Loading file...")

    loader = PyPDFLoader(document_path)
    documents = loader.load()

    print(f"✓ Loaded {len(documents)} pages from file")

    #--------------------------------------------------------------------------------
    # STEP 2: AUTO-DETECT TOPIC
    #--------------------------------------------------------------------------------
    print(f"\n[2/6] Auto-detecting document topic...")

    detected_topic = detect_document_topic(documents)

    print(f"✓ Topic auto-detected: '{detected_topic}'")

    #--------------------------------------------------------------------------------
    # STEP 3: PREPROCESSING
    #--------------------------------------------------------------------------------
    print(f"\n[3/6] Applying text preprocessing...")

    for doc in documents:
      doc.page_content = clean_text(doc.page_content)

    print(f"✓ Cleaned {len(documents)} pages")

    #--------------------------------------------------------------------------------
    # STEP 4: CHUNKING
    #--------------------------------------------------------------------------------
    print(f"\n[4/6] Splitting documents into chunks...")

    text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=1000,
      chunk_overlap=200
    )

    chunks = text_splitter.split_documents(documents)

    print(f"✓ Split {len(chunks)} chunks")

    #--------------------------------------------------------------------------------
    # STEP 5: ADD METADATA
    #--------------------------------------------------------------------------------
    print(f"\n[5/6] Enriching documents with metadata...")

    for chunk in chunks:
      chunk.metadata.update({
          'topic': detected_topic,
      })

    print(f"✓ Metadata enriched for all documents")

    #--------------------------------------------------------------------------------
    # STEP 6: Use Base Retriever (NEW IN v5 Build)
    #--------------------------------------------------------------------------------
    print(f"\n[6/6] ParentDocumentRetriever processing...")

    base_retriever.add_documents(chunks)
    
    print(f"✓ Stored")

    return len(chunks), detected_topic

In [ ]:
ingest_documents("https://bitcoin.org/bitcoin.pdf")

ingest_documents("../assets/ethereum.pdf")

## Inference

In [ ]:
#@title RAG Chain Creation

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain():
    """
    CREATE RAG CHAIN

    Returns:
        RunnableSequence: RAG chain
    """

    rag_template = ChatPromptTemplate.from_template(
      """
      You are a helpful assistant answering questions about cryptocurrency whitepapers.

      Use the conversation history and provided documents to answer the current question.
      If the question references previous context (e.g., "it", "this", "that"), use the conversation history to understand what is being referenced.

      Instructions:
      - Answer clearly and concisely (max 2-3 paragraphs)
      - Use only the provided documents as context
      - If the question is a follow-up, maintain conversation continuity
      - If the answer is not in the documents, say "I don't have enough information to answer that question"

      Conversation History:
      {history}

      Documents:
      {context}

      Current Question: {query}

      Answer:
      """
    )

    rag_chain = rag_template | llm | StrOutputParser()

    return rag_chain

In [ ]:
#@title Format Chat History Helper (NEW IN v4 Build)

def format_chat_history(history: list, max_turns: int = 5) -> str:
    """
    FORMAT CHAT HISTORY

    Converts Gradio's messages format history into a readable string.

    WHY LIMIT TURNS?
    - Token limits: Long conversations exceed context window
    - Relevance: Recent turns are more relevant
    - Cost: Fewer tokens = lower cost
    - Performance: Faster processing

    Args:
        history (list): Gradio chat history in messages format
        max_turns (int): Maximum number of conversation turns to include

    Returns:
        str: Formatted conversation history
    """

    if not history:
        return "No previous conversation."

    # Limit to last N turns (each turn = user + assistant message)
    recent_history = history[-(max_turns * 2):]

    # Format as readable conversation
    formatted = []
    for message in recent_history:
        role = message["role"].capitalize()
        content = message["content"]
        formatted.append(f"{role}: {content}")

    return "\n".join(formatted)

In [ ]:
#@title Inference with MultiQuery + Reranking (NEW IN v5 Build)

from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

def inference(
    query: str,
    chat_history: list = None
) -> str:
    """
    INFERENCE WITH MULTI-QUERY RETRIEVAL + RERANKING - NEW IN v5 Build

    MULTI-QUERY:
    - LLM generates query variations
    - Each variation searches the vector store
    - Combines and deduplicates results

    RERANKING:
    - Cross-encoder scores each doc against ORIGINAL query
    - Reranks all documents by relevance
    - Keeps only top_n most relevant

    TRADE-OFFS:
    - Slower (LLM calls for query expansion + reranking)
    - More expensive (more tokens processed)
    - BUT: Maximum quality for complex/ambiguous queries
    """

    print("="*80)
    print(f"RUNNING MULTI-QUERY INFERENCE - VERSION {VERSION} Build")
    print("="*80)

    #--------------------------------------------------------------------------------
    # STEP 1: FORMAT CONVERSATION HISTORY
    #--------------------------------------------------------------------------------
    print(f"\n[1/4] Formatting conversation history...")
    formatted_history = format_chat_history(chat_history, max_turns=5)
    print(f"✓ History formatted ({len(chat_history or [])} messages)")

    #--------------------------------------------------------------------------------
    # STEP 2: MULTI-QUERY RETRIEVAL + RERANKING
    #--------------------------------------------------------------------------------
    print(f"\n[2/4] Performing multi-query retrieval + reranking...")

    # Multi-query retriever: generates query variations
    multiquery_retriever = MultiQueryRetriever.from_llm(
      retriever=base_retriever,
      llm=llm
    )

    # Reranker: cross-encoder scoring
    compressor = FlashrankRerank(
        top_n=4
    )

    # Compression retriever: combines multiquery + reranking
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=multiquery_retriever
    )

    results = compression_retriever.invoke(query)

    print(f"✓ Retrieved and reranked {len(results)} documents")

    #--------------------------------------------------------------------------------
    # STEP 3: FORMAT CONTEXT
    #--------------------------------------------------------------------------------
    print(f"\n[3/4] Formatting context for LLM...")
    context = "\n\n".join([doc.page_content for doc in results])

    #--------------------------------------------------------------------------------
    # STEP 4: GENERATE ANSWER
    #--------------------------------------------------------------------------------
    print(f"\n[4/4] Invoking RAG chain...")

    response = create_rag_chain().invoke({
        "context": context,
        "query": query,
        "history": formatted_history
    })

    print("\n" + "="*80)
    print("INFERENCE COMPLETE")
    print("="*80)

    return response

In [ ]:
# Test multi-query inference
inference("What is Bitcoin?")

## Gradio Demo

In [ ]:
def chat_inference(message, history):
  """Gradio ChatInterface wrapper"""
  return inference(query=message, chat_history=history)

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chat_inference,
    title="Crypto RAG Assistant (v5)",
    description="Ask questions about crypto. Features: memory + multi-query + reranking.",
    examples=[
        "What is Bitcoin?",
        "How does it work?",
        "What is proof of work?",
        "Explain the blockchain structure"
    ],
)

demo.launch(share=True, debug=True)